## Load and Prepare Data




In [1]:
import pandas as pd

# Load the 'data train manual' sheet into df_train_raw
df_train_raw = pd.read_excel('DataSet.xlsx', sheet_name='data train manual')

# Load the 'data test' sheet into df_test_raw
df_test_raw = pd.read_excel('DataSet.xlsx', sheet_name='data test')

print("Data loaded successfully. Displaying first 5 rows of df_train_raw and df_test_raw:")
print("df_train_raw head:")
print(df_train_raw.head())
print("\ndf_test_raw head:")
print(df_test_raw.head())

Data loaded successfully. Displaying first 5 rows of df_train_raw and df_test_raw:
df_train_raw head:
                                             positif  \
0                 everything alex absolutely perfect   
1  everything skinny british man perfect thank ev...   
2                 hes little secret anymore hes good   
3                                      peaceful song   
4                         song criminally underrated   

                                             negatif  \
0  literally one artistes sounds like studio vers...   
1       lost dad last december hadn t time cry today   
2                                        cry singing   
3                                         i m crying   
4  song beautiful encapsulates like grow without ...   

                                              netral  
0  shoulder it s like understands everything i ve...  
1   songs simultaneously therapeutic depressing time  
2                  therapy expensivebrthis song free  
3  i

In [2]:
training_data = []
for col in ['positif', 'negatif', 'netral']:
    temp_df = df_train_raw[df_train_raw[col].notna()].copy()
    if not temp_df.empty:
        for comment in temp_df[col]:
            training_data.append({'comment': comment, 'label': col})

df_train = pd.DataFrame(training_data)

print("Training data consolidated successfully. Displaying first 5 rows of df_train:")
print(df_train.head())
print("\nValue counts for labels in df_train:")
print(df_train['label'].value_counts())

Training data consolidated successfully. Displaying first 5 rows of df_train:
                                             comment    label
0                 everything alex absolutely perfect  positif
1  everything skinny british man perfect thank ev...  positif
2                 hes little secret anymore hes good  positif
3                                      peaceful song  positif
4                         song criminally underrated  positif

Value counts for labels in df_train:
label
positif    100
negatif    100
netral      50
Name: count, dtype: int64


In [3]:
df_test = pd.DataFrame()
df_test['comment'] = df_test_raw['cleaned_text']

print("Test data comment column identified. Displaying first 5 rows of df_test:")
print(df_test.head())

Test data comment column identified. Displaying first 5 rows of df_test:
                                             comment
0  everything skinny british man perfect thank ev...
1                                 makes easy cry lt3
2                                     makes easy cry
3              rex bombing us content last days love
4          song doesnt needs many views made legends


In [5]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True) # Added this line to download punkt_tab

def preprocess_text(text):
    if not isinstance(text, str):
        return text # Return non-string values as is, or handle as error

    # Convert to lowercase
    text = text.lower()

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Tokenize words
    words = word_tokenize(text)

    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    filtered_words = [word for word in words if word not in stop_words]

    # Join back into a string
    text = ' '.join(filtered_words)
    return text

df_train['comment'] = df_train['comment'].apply(preprocess_text)
df_test['comment'] = df_test['comment'].apply(preprocess_text)

print("Text preprocessing completed for df_train and df_test. Displaying first 5 rows of preprocessed df_train:")
print(df_train.head())
print("\nDisplaying first 5 rows of preprocessed df_test:")
print(df_test.head())

Text preprocessing completed for df_train and df_test. Displaying first 5 rows of preprocessed df_train:
                                             comment    label
0                 everything alex absolutely perfect  positif
1  everything skinny british man perfect thank ev...  positif
2                 hes little secret anymore hes good  positif
3                                      peaceful song  positif
4                         song criminally underrated  positif

Displaying first 5 rows of preprocessed df_test:
                                             comment
0  everything skinny british man perfect thank ev...
1                                 makes easy cry lt3
2                                     makes easy cry
3              rex bombing us content last days love
4          song doesnt needs many views made legends


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    df_train['comment'],
    df_train['label'],
    test_size=0.2,
    random_state=42,
    stratify=df_train['label']
)

print("Training and validation sets created successfully.")
print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_val shape: {y_val.shape}")

print("\nDistribution of labels in y_train:")
print(y_train.value_counts(normalize=True))
print("\nDistribution of labels in y_val:")
print(y_val.value_counts(normalize=True))

Training and validation sets created successfully.
X_train shape: (200,)
X_val shape: (50,)
y_train shape: (200,)
y_val shape: (50,)

Distribution of labels in y_train:
label
negatif    0.4
positif    0.4
netral     0.2
Name: proportion, dtype: float64

Distribution of labels in y_val:
label
positif    0.4
negatif    0.4
netral     0.2
Name: proportion, dtype: float64


## Define Experiment Configurations



In [7]:
import random

# 1. Define a list named `feature_extractors`
feature_extractors = ['TF-IDF', 'Word2Vec', 'BoW', 'GloVe']

# 2. Define a list named `classifiers`
classifiers = ['Naive Bayes', 'SVM', 'Logistic Regression', 'LSTM', 'Transformer']

# 4. Create an empty list called `experiment_configurations`
experiment_configurations = []

# 5. Iterate 4 times to select unique combinations
while len(experiment_configurations) < 4:
    # a. Randomly select one feature extractor
    selected_feature_extractor = random.choice(feature_extractors)
    # b. Randomly select one classifier
    selected_classifier = random.choice(classifiers)

    # c. Create a tuple with the selected feature extractor and classifier
    new_combination = (selected_feature_extractor, selected_classifier)

    # d. Add this tuple to `experiment_configurations` only if it's not already present
    # e. If a duplicate is selected, retry the selection for that iteration until a unique pair is found.
    if new_combination not in experiment_configurations:
        experiment_configurations.append(new_combination)

# 6. Print the `experiment_configurations` list
print("Selected Experiment Configurations:")
for i, config in enumerate(experiment_configurations):
    print(f"Experiment {i+1}: Feature Extractor: {config[0]}, Classifier: {config[1]}")

Selected Experiment Configurations:
Experiment 1: Feature Extractor: TF-IDF, Classifier: SVM
Experiment 2: Feature Extractor: TF-IDF, Classifier: Transformer
Experiment 3: Feature Extractor: GloVe, Classifier: SVM
Experiment 4: Feature Extractor: TF-IDF, Classifier: LSTM


## Experiment 1: Run First Model Combination
**(TF-IDF + SVM)**


In [8]:
results = {}
test_predictions = {}

# Extract the first experiment configuration
current_experiment_config = experiment_configurations[0]
feature_extractor_name = current_experiment_config[0]
classifier_name = current_experiment_config[1]

print(f"Starting Experiment 1: Feature Extractor: {feature_extractor_name}, Classifier: {classifier_name}")

Starting Experiment 1: Feature Extractor: TF-IDF, Classifier: SVM


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limiting features to avoid very sparse matrices and memory issues

# Fit on training data and transform X_train, X_val, and df_test['comment']
X_train_features = tfidf_vectorizer.fit_transform(X_train)
X_val_features = tfidf_vectorizer.transform(X_val)
X_test_features = tfidf_vectorizer.transform(df_test['comment'])

print("TF-IDF feature extraction completed.")
print(f"X_train_features shape: {X_train_features.shape}")
print(f"X_val_features shape: {X_val_features.shape}")
print(f"X_test_features shape: {X_test_features.shape}")

TF-IDF feature extraction completed.
X_train_features shape: (200, 941)
X_val_features shape: (50, 941)
X_test_features shape: (760, 941)


In [11]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

# Initialize and train the SVC model
svm_model = SVC(random_state=42)
svm_model.fit(X_train_features, y_train)

# Predict on the validation set
y_val_pred = svm_model.predict(X_val_features)

# Evaluate performance on the validation set
print(f"\n--- Evaluation for {feature_extractor_name} + {classifier_name} ---")
print("Classification Report on Validation Set:")
print(classification_report(y_val, y_val_pred, zero_division=0))

accuracy = accuracy_score(y_val, y_val_pred)
print(f"Accuracy on Validation Set: {accuracy:.4f}")

# Store accuracy in results dictionary
results[f'{feature_extractor_name}_{classifier_name}_accuracy'] = accuracy

# Predict on the test set
y_test_pred = svm_model.predict(X_test_features)

# Store test predictions in test_predictions dictionary
test_predictions[f'{feature_extractor_name}_{classifier_name}_predictions'] = y_test_pred

print(f"\nTest predictions for {feature_extractor_name} + {classifier_name} generated and stored.")
print("First 5 test predictions:")
print(y_test_pred[:5])


--- Evaluation for TF-IDF + SVM ---
Classification Report on Validation Set:
              precision    recall  f1-score   support

     negatif       0.79      0.75      0.77        20
      netral       0.00      0.00      0.00        10
     positif       0.58      0.90      0.71        20

    accuracy                           0.66        50
   macro avg       0.46      0.55      0.49        50
weighted avg       0.55      0.66      0.59        50

Accuracy on Validation Set: 0.6600

Test predictions for TF-IDF + SVM generated and stored.
First 5 test predictions:
['positif' 'negatif' 'negatif' 'positif' 'positif']


## Experiment 2: Run Second Model Combination
**(TF-IDF + Transformer)**


In [12]:
import numpy as np
from sklearn.preprocessing import LabelEncoder
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense
from sklearn.metrics import classification_report, accuracy_score

# Extract the second experiment configuration
current_experiment_config = experiment_configurations[1]
feature_extractor_name = current_experiment_config[0]
classifier_name = current_experiment_config[1]

print(f"\nStarting Experiment 2: Feature Extractor: {feature_extractor_name}, Classifier: {classifier_name}")

# Reuse already computed TF-IDF features (X_train_features, X_val_features, X_test_features)
# Convert sparse matrices to dense arrays for deep learning models
X_train_features_dense = X_train_features.toarray()
X_val_features_dense = X_val_features.toarray()
X_test_features_dense = X_test_features.toarray()

print("Converted TF-IDF features to dense arrays.")
print(f"X_train_features_dense shape: {X_train_features_dense.shape}")
print(f"X_val_features_dense shape: {X_val_features_dense.shape}")
print(f"X_test_features_dense shape: {X_test_features_dense.shape}")

# Prepare categorical labels for deep learning model
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)

y_train_one_hot = to_categorical(y_train_encoded)
y_val_one_hot = to_categorical(y_val_encoded)

print("Labels converted to numerical and one-hot encoded.")
print(f"Number of unique classes: {len(label_encoder.classes_)}")
print(f"y_train_one_hot shape: {y_train_one_hot.shape}")
print(f"y_val_one_hot shape: {y_val_one_hot.shape}")

# Build the MLP model (acting as 'Transformer' in this context)
model = Sequential()
model.add(Dense(128, activation='relu', input_shape=(X_train_features_dense.shape[1],)))
model.add(Dense(64, activation='relu'))
model.add(Dense(len(label_encoder.classes_), activation='softmax'))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("MLP model built and compiled. Training started...")

# Train the model
history = model.fit(
    X_train_features_dense, y_train_one_hot,
    epochs=10, # You might adjust this based on performance
    batch_size=32,
    validation_data=(X_val_features_dense, y_val_one_hot),
    verbose=0 # Set to 1 for progress bar
)

print("MLP model training completed.")

# Evaluate performance on the validation set
y_val_pred_probabilities = model.predict(X_val_features_dense, verbose=0)
y_val_pred_encoded = np.argmax(y_val_pred_probabilities, axis=1)
y_val_pred_decoded = label_encoder.inverse_transform(y_val_pred_encoded)

print(f"\n--- Evaluation for {feature_extractor_name} + {classifier_name} ---")
print("Classification Report on Validation Set:")
print(classification_report(y_val, y_val_pred_decoded, zero_division=0))

accuracy = accuracy_score(y_val, y_val_pred_decoded)
print(f"Accuracy on Validation Set: {accuracy:.4f}")

# Store accuracy in results dictionary
results[f'{feature_extractor_name}_{classifier_name}_accuracy'] = accuracy

# Generate predictions for the test set
y_test_pred_probabilities = model.predict(X_test_features_dense, verbose=0)
y_test_pred_encoded = np.argmax(y_test_pred_probabilities, axis=1)
y_test_pred_decoded = label_encoder.inverse_transform(y_test_pred_encoded)

# Store test predictions in test_predictions dictionary
test_predictions[f'{feature_extractor_name}_{classifier_name}_predictions'] = y_test_pred_decoded

print(f"\nTest predictions for {feature_extractor_name} + {classifier_name} generated and stored.")
print("First 5 test predictions:")
print(y_test_pred_decoded[:5])


Starting Experiment 2: Feature Extractor: TF-IDF, Classifier: Transformer
Converted TF-IDF features to dense arrays.
X_train_features_dense shape: (200, 941)
X_val_features_dense shape: (50, 941)
X_test_features_dense shape: (760, 941)
Labels converted to numerical and one-hot encoded.
Number of unique classes: 3
y_train_one_hot shape: (200, 3)
y_val_one_hot shape: (50, 3)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


MLP model built and compiled. Training started...
MLP model training completed.

--- Evaluation for TF-IDF + Transformer ---
Classification Report on Validation Set:
              precision    recall  f1-score   support

     negatif       0.80      0.80      0.80        20
      netral       1.00      0.10      0.18        10
     positif       0.62      0.90      0.73        20

    accuracy                           0.70        50
   macro avg       0.81      0.60      0.57        50
weighted avg       0.77      0.70      0.65        50

Accuracy on Validation Set: 0.7000

Test predictions for TF-IDF + Transformer generated and stored.
First 5 test predictions:
['positif' 'negatif' 'negatif' 'positif' 'positif']


## Experiment 3: Run 3rd Model Combination
**(GloVe + SVM)**

In [14]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 50.4 MB/s eta 0:00:00


In [15]:
import gensim.downloader as api
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

# Extract the third experiment configuration
current_experiment_config = experiment_configurations[2]
feature_extractor_name = current_experiment_config[0]
classifier_name = current_experiment_config[1]

print(f"\nStarting Experiment 3: Feature Extractor: {feature_extractor_name}, Classifier: {classifier_name}")

# Load pre-trained GloVe embeddings (if not already loaded)
# This might take some time the first time it's run
try:
    glove_vectors = api.load("glove-wiki-gigaword-50") # Using 50-dimensional vectors for speed
    print("GloVe embeddings loaded successfully.")
except ValueError as e:
    print(f"Error loading GloVe: {e}. Attempting to download first.")
    # Attempt to download if not found locally, though api.load usually handles this
    api.load("glove-wiki-gigaword-50", return_path=True) # Just to trigger download
    glove_vectors = api.load("glove-wiki-gigaword-50")
    print("GloVe embeddings downloaded and loaded successfully.")

# Function to convert text to GloVe vectors (averaging word vectors)
def text_to_glove_vector(text, model):
    words = text.split()
    vector_list = []
    for word in words:
        if word in model:
            vector_list.append(model[word])
    if len(vector_list) == 0:
        return np.zeros(model.vector_size) # Return a zero vector if no words are found
    return np.mean(vector_list, axis=0)

# Apply GloVe feature extraction
X_train_glove = np.array([text_to_glove_vector(text, glove_vectors) for text in X_train])
X_val_glove = np.array([text_to_glove_vector(text, glove_vectors) for text in X_val])
X_test_glove = np.array([text_to_glove_vector(text, glove_vectors) for text in df_test['comment']])

print("GloVe feature extraction completed.")
print(f"X_train_glove shape: {X_train_glove.shape}")
print(f"X_val_glove shape: {X_val_glove.shape}")
print(f"X_test_glove shape: {X_test_glove.shape}")

# Initialize and train the SVC model
svm_model_glove = SVC(random_state=42)
svm_model_glove.fit(X_train_glove, y_train)

# Predict on the validation set
y_val_pred_glove = svm_model_glove.predict(X_val_glove)

# Evaluate performance on the validation set
print(f"\n--- Evaluation for {feature_extractor_name} + {classifier_name} ---")
print("Classification Report on Validation Set:")
print(classification_report(y_val, y_val_pred_glove, zero_division=0))

accuracy_glove = accuracy_score(y_val, y_val_pred_glove)
print(f"Accuracy on Validation Set: {accuracy_glove:.4f}")

# Store accuracy in results dictionary
results[f'{feature_extractor_name}_{classifier_name}_accuracy'] = accuracy_glove

# Predict on the test set
y_test_pred_glove = svm_model_glove.predict(X_test_glove)

# Store test predictions in test_predictions dictionary
test_predictions[f'{feature_extractor_name}_{classifier_name}_predictions'] = y_test_pred_glove

print(f"\nTest predictions for {feature_extractor_name} + {classifier_name} generated and stored.")
print("First 5 test predictions:")
print(y_test_pred_glove[:5])


Starting Experiment 3: Feature Extractor: GloVe, Classifier: SVM
[==================================================] 100.0% 66.0/66.0MB downloaded
GloVe embeddings loaded successfully.
GloVe feature extraction completed.
X_train_glove shape: (200, 50)
X_val_glove shape: (50, 50)
X_test_glove shape: (760, 50)

--- Evaluation for GloVe + SVM ---
Classification Report on Validation Set:
              precision    recall  f1-score   support

     negatif       0.61      0.95      0.75        20
      netral       0.00      0.00      0.00        10
     positif       0.68      0.65      0.67        20

    accuracy                           0.64        50
   macro avg       0.43      0.53      0.47        50
weighted avg       0.52      0.64      0.56        50

Accuracy on Validation Set: 0.6400

Test predictions for GloVe + SVM generated and stored.
First 5 test predictions:
['positif' 'negatif' 'negatif' 'negatif' 'positif']


## Experiment 4: Run 4th Model Combination
**(TF-IDF + LSTM)**

In [16]:
from keras.models import Sequential
from keras.layers import LSTM, Dense
from sklearn.metrics import classification_report, accuracy_score
import numpy as np
from sklearn.preprocessing import LabelEncoder
from keras.utils import to_categorical

# Extract the fourth experiment configuration
current_experiment_config = experiment_configurations[3]
feature_extractor_name = current_experiment_config[0]
classifier_name = current_experiment_config[1]

print(f"\nStarting Experiment 4: Feature Extractor: {feature_extractor_name}, Classifier: {classifier_name}")

# Reuse already computed TF-IDF features (X_train_features, X_val_features, X_test_features)
# Convert sparse matrices to dense arrays for deep learning models
X_train_features_dense = X_train_features.toarray()
X_val_features_dense = X_val_features.toarray()
X_test_features_dense = X_test_features.toarray()

# Reshape data for LSTM input: (samples, timesteps, features)
# Treating each document's TF-IDF vector as a single timestep
X_train_reshaped = X_train_features_dense.reshape(X_train_features_dense.shape[0], 1, X_train_features_dense.shape[1])
X_val_reshaped = X_val_features_dense.reshape(X_val_features_dense.shape[0], 1, X_val_features_dense.shape[1])
X_test_reshaped = X_test_features_dense.reshape(X_test_features_dense.shape[0], 1, X_test_features_dense.shape[1])

print("Reshaped TF-IDF features for LSTM.")
print(f"X_train_reshaped shape: {X_train_reshaped.shape}")
print(f"X_val_reshaped shape: {X_val_reshaped.shape}")
print(f"X_test_reshaped shape: {X_test_reshaped.shape}")

# Prepare categorical labels for deep learning model (reuse from Experiment 2)
# If label_encoder was not created in Experiment 2, create it here.
if 'label_encoder' not in locals():
    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train)
    y_val_encoded = label_encoder.transform(y_val)
else:
    y_train_encoded = label_encoder.transform(y_train)
    y_val_encoded = label_encoder.transform(y_val)

y_train_one_hot = to_categorical(y_train_encoded)
y_val_one_hot = to_categorical(y_val_encoded)

print("Labels converted to numerical and one-hot encoded.")

# Build the LSTM model
model_lstm = Sequential()
model_lstm.add(LSTM(128, input_shape=(X_train_reshaped.shape[1], X_train_reshaped.shape[2])))
model_lstm.add(Dense(len(label_encoder.classes_), activation='softmax'))

model_lstm.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("LSTM model built and compiled. Training started...")

# Train the model
history_lstm = model_lstm.fit(
    X_train_reshaped, y_train_one_hot,
    epochs=10,
    batch_size=32,
    validation_data=(X_val_reshaped, y_val_one_hot),
    verbose=0
)

print("LSTM model training completed.")

# Evaluate performance on the validation set
y_val_pred_probabilities_lstm = model_lstm.predict(X_val_reshaped, verbose=0)
y_val_pred_encoded_lstm = np.argmax(y_val_pred_probabilities_lstm, axis=1)
y_val_pred_decoded_lstm = label_encoder.inverse_transform(y_val_pred_encoded_lstm)

print(f"\n--- Evaluation for {feature_extractor_name} + {classifier_name} ---")
print("Classification Report on Validation Set:")
print(classification_report(y_val, y_val_pred_decoded_lstm, zero_division=0))

accuracy_lstm = accuracy_score(y_val, y_val_pred_decoded_lstm)
print(f"Accuracy on Validation Set: {accuracy_lstm:.4f}")

# Store accuracy in results dictionary
results[f'{feature_extractor_name}_{classifier_name}_accuracy'] = accuracy_lstm

# Generate predictions for the test set
y_test_pred_probabilities_lstm = model_lstm.predict(X_test_reshaped, verbose=0)
y_test_pred_encoded_lstm = np.argmax(y_test_pred_probabilities_lstm, axis=1)
y_test_pred_decoded_lstm = label_encoder.inverse_transform(y_test_pred_encoded_lstm)

# Store test predictions in test_predictions dictionary
test_predictions[f'{feature_extractor_name}_{classifier_name}_predictions'] = y_test_pred_decoded_lstm

print(f"\nTest predictions for {feature_extractor_name} + {classifier_name} generated and stored.")
print("First 5 test predictions:")
print(y_test_pred_decoded_lstm[:5])



Starting Experiment 4: Feature Extractor: TF-IDF, Classifier: LSTM
Reshaped TF-IDF features for LSTM.
X_train_reshaped shape: (200, 1, 941)
X_val_reshaped shape: (50, 1, 941)
X_test_reshaped shape: (760, 1, 941)
Labels converted to numerical and one-hot encoded.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


LSTM model built and compiled. Training started...
LSTM model training completed.

--- Evaluation for TF-IDF + LSTM ---
Classification Report on Validation Set:
              precision    recall  f1-score   support

     negatif       0.84      0.80      0.82        20
      netral       0.00      0.00      0.00        10
     positif       0.55      0.85      0.67        20

    accuracy                           0.66        50
   macro avg       0.46      0.55      0.50        50
weighted avg       0.56      0.66      0.59        50

Accuracy on Validation Set: 0.6600

Test predictions for TF-IDF + LSTM generated and stored.
First 5 test predictions:
['positif' 'negatif' 'negatif' 'negatif' 'positif']


In [19]:
import pandas as pd

# Convert results dictionary to a DataFrame for better comparison
accuracy_df = pd.DataFrame(results.items(), columns=['Model', 'Accuracy'])
accuracy_df['Feature Extractor'] = accuracy_df['Model'].apply(lambda x: x.split('_')[0])
accuracy_df['Classifier'] = accuracy_df['Model'].apply(lambda x: x.split('_')[1])
accuracy_df = accuracy_df[['Feature Extractor', 'Classifier', 'Accuracy']]

print("\n--- Model Accuracy Comparison ---")
print(accuracy_df)



--- Model Accuracy Comparison ---
  Feature Extractor   Classifier  Accuracy
0            TF-IDF          SVM      0.66
1            TF-IDF  Transformer      0.70
2             GloVe          SVM      0.64
3            TF-IDF         LSTM      0.66


In [20]:
print("\n--- Test Predictions (First 5 for Each Model) ---")
for model_key, predictions in test_predictions.items():
    parts = model_key.split('_')
    feature_extractor = parts[0]
    classifier = parts[1]
    print(f'Predictions for {feature_extractor} + {classifier}: {predictions[:5]}')



--- Test Predictions (First 5 for Each Model) ---
Predictions for TF-IDF + SVM: ['positif' 'negatif' 'negatif' 'positif' 'positif']
Predictions for TF-IDF + Transformer: ['positif' 'negatif' 'negatif' 'positif' 'positif']
Predictions for GloVe + SVM: ['positif' 'negatif' 'negatif' 'negatif' 'positif']
Predictions for TF-IDF + LSTM: ['positif' 'negatif' 'negatif' 'negatif' 'positif']
